
# Stream Customers Data From Cloud Files to Delta Lake

1. Read files from cloud storage using DataStreamReader API
2. Transform the dataframe to add the following columns
    1. file path: Cloud file path
    2. ingestion data : Current Timestamp
3. Write the transformed data stream to Delta Lake Table


## 1. Read files using DataStreamReader API

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (StructType, 
                               StructField, 
                               IntegerType, 
                               StringType,
                               DateType,
                               TimestampType,  
                               FloatType)
                

In [0]:
# Initialize the Stream

jsonSchema = StructType([
        StructField('customer_id', IntegerType()), 
        StructField('customer_name', StringType()), 
        StructField('date_of_birth', DateType()), 
        StructField('telephone', StringType()), 
        StructField('email', StringType()), 
        StructField('member_since', DateType()),
        StructField('created_timestamp', TimestampType())]
)


streamingInputDF = (spark.readStream
        .format('json')
        .schema(jsonSchema) # Set the schema of the JSON data
        .option('maxFilesPerTrigger', 1) # Treat a sequence of files as a stream by picking one file at a time
        .load('/Volumes/gizmobox/landing/operational_data/customer_stream/'))


## 2. Transform the dataframe to add the following columns

  1. file path : Cloud file path
  2. ingestion date : Current Timestamp

In [0]:
streamingInputDF = (streamingInputDF.withColumn('filepath', F.col("_metadata.file_path"))
                .withColumn('ingestion_date', F.current_date()))



## 3. Write the transformed data stream to Delta Table

In [0]:
# Start a stream job
streamingOutputDF = (streamingInputDF.writeStream
                .format("delta")
                .trigger(once = True)
                .outputMode("append")
                .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customer_stream/_checkpoint_stream")
                .toTable("gizmobox.bronze.customers_stream"))

In [0]:
%sql

SELECT member_since, count(*) as count FROM gizmobox.bronze.customers_stream
GROUP BY member_since

In [0]:
streamingInputDF = (streamingInputDF.groupBy('member_since')
                 .agg(F.count('customer_id').alias('cnt_customer')))

In [0]:
# Start a stream job
streamingOutputDF = (streamingInputDF.writeStream
                .format("delta")
                .trigger(once = True)
                .outputMode("complete")
                .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customer_stream_groupby/_checkpoint_stream")
                .toTable("gizmobox.bronze.customers_stream_groupby"))

In [0]:
%sql

SELECT * FROM gizmobox.bronze.customers_stream_groupby;




Yes — here are **official sources** that clearly outline the Structured Streaming output mode rules and the limitations that cause the errors you’re seeing:

---

## 📌 1) Spark Structured Streaming Official Docs — Output Modes

The Spark Programming Guide (official Apache Spark docs) defines the three output modes and when they’re allowed:

### Output Modes

* **Append** — Writes only new rows. Supported when existing rows won’t change (i.e., no aggregations that update old results). ([Apache Spark][1])
* **Complete** — Writes *all* rows of the result table each trigger. Only supported for **aggregation queries**. ([Apache Spark][1])
* **Update** — Writes only rows that *changed* since the last trigger. ([Apache Spark][1])

📌 The **compatibility matrix** explicitly shows that:

* Queries **without aggregations** support *Append* and sometimes *Update*, but **not Complete**. ([Apache Spark][1])
* Queries **with aggregations** support *Complete* (and *Update*). ([Apache Spark][1])

So trying to use **Complete** on a *non-aggregation* query leads to:

```
Invalid streaming output mode: complete
```

because Spark does not allow Complete there. ([Apache Spark][1])

---

## 📌 2) Delta Lake Connector (DeltaDataSource)

This documentation points out the exact reason for your *first* error:

> The Delta sink only supports **Append** and **Complete** output modes.
> If you pass any other output mode (such as `Update`), you’ll see:
>
> ```
> Data source ... does not support [outputMode] output mode
> ```
>
> because DeltaDataSource rejects it. ([japila-books][2])

This is exactly why:

```
…DeltaDataSource does not support Update output mode…
```

appears — *Update* isn’t supported by Delta as a sink. ([japila-books][2])

---

## 🎯 Summary of the Rules (From Documentation)

| Streaming Query Type | Valid Output Modes (Spark)               | Note                                            |
| -------------------- | ---------------------------------------- | ----------------------------------------------- |
| No aggregation       | `append`, sometimes `update`             | **Complete not allowed** ([Apache Spark][1])    |
| Aggregation          | `complete`, `update`, sometimes `append` | Allows full results. ([Apache Spark][1])        |
| **Delta Sink**       | `append`, `complete` only                | *Update rejected* by Delta. ([japila-books][2]) |

---

If you want **direct links to the official docs** above, let me know and I can share them too.

[1]: https://spark.apache.org/docs/_site/streaming/apis-on-dataframes-and-datasets.html?utm_source=chatgpt.com "Structured Streaming Programming Guide - Spark 4.1.0-preview2 Documentation"
[2]: https://books.japila.pl/delta-lake-internals/spark-connector/DeltaDataSource/?utm_source=chatgpt.com "DeltaDataSource - The Internals of Delta Lake"
